# 11 · Data quality checks and reconciliation (BigQuery)

A pipeline must **prove** its data is right. These SQL checks are the same ones you put in a
data quality step (in Composer/Airflow, dbt tests, or Great Expectations).

Good checks return **a number you can compare**: 0 bad rows = pass.

- completeness: NULLs where there should be none
- uniqueness: duplicate keys
- referential integrity: child rows whose parent is missing (**orphans**)
- validity: values outside the allowed range/list
- **reconciliation**: source and target have the same row count and the same SUM

**How to run:** Start each SQL cell with `%%bigquery`. Write tables as `practice.table_name`.
Each cell prints how much data it read. `%%bigquery dry` checks the size for free.

Tables: see [tables_guide.md](../../cheatsheets/tables_guide.md). Run the setup cell first (Shift+Enter).

In [ ]:
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "settings.py").exists())   # the repo folder
sys.path.insert(0, str(ROOT / "bigquery"))
%load_ext bq_helper

## Q1. NULL profile of a table  ·  🟢 easy

For `orders`, in one row show `total_rows` and the number of NULLs in each column: `null_order_id`, `null_customer_id`, `null_order_date`, `null_amount`, `null_status`.

*Expected result: 1 row · columns: `total_rows`, `null_order_id`, `null_customer_id`, `null_order_date`, `null_amount`, `null_status`*

In [ ]:
%%bigquery
-- your answer here

## Q2. Is the key unique?  ·  🟢 easy

Check if `order_id` is a unique key in `orders`. Show `dup_keys` = how many order_ids appear more than once, and `extra_rows` = how many rows too many.

*Expected result: 1 row · columns: `dup_keys`, `extra_rows`*

In [ ]:
%%bigquery
-- your answer here

## Q3. Orphan rows (referential integrity)  ·  🟡 medium

Make one test-report query with UNION ALL. Rows: `check_name`, `bad_rows`:
- `items_without_order`: order_items whose order_id is not in orders
- `items_without_product`: order_items whose product_id is not in products
- `employee_bad_dept`: employees whose dept_id is **not NULL** but not in departments
- `employee_bad_manager`: employees whose manager_id is not NULL but not an emp_id

*Expected result: 4 rows · columns: `check_name`, `bad_rows`*

In [ ]:
%%bigquery
-- your answer here

## Q4. Validity rules  ·  🟡 medium

Count rows of `orders` that break each rule. One row with: `negative_amount` (amount < 0), `huge_amount` (amount > 1000), `future_date` (order_date after 2025-12-31), `bad_status` (status not in completed / cancelled / pending / returned, or NULL).

*Expected result: 1 row · columns: `negative_amount`, `huge_amount`, `future_date`, `bad_status`*

In [ ]:
%%bigquery
-- your answer here

## Q5. Reconciliation: source vs target  ·  🟡 medium

The target is a daily summary built from completed orders. Compare it with the source:
`src_rows`, `tgt_rows` (number of orders in each), `src_amount`, `tgt_amount`, and `result` = `MATCH` if both pairs are equal, else `MISMATCH`.

*Expected result: 1 row · columns: `src_rows`, `tgt_rows`, `src_amount`, `tgt_amount`, `result`*

In [ ]:
%%bigquery
-- your answer here

## Q6. Row-level differences between source and target  ·  🔴 hard

Using the `src` and `tgt` CTEs given in the answer cell (tgt is missing order 1005 and has a wrong amount for 1010), list `order_id` and `issue` = `missing_in_target`, `missing_in_source` or `amount_differs`. Careful: some amounts are NULL on both sides: that is **not** a difference.

<details><summary>Hint</summary>

FULL OUTER JOIN on order_id. Then compare amounts with a NULL-safe test.

</details>

*Expected result: 2 rows · columns: `order_id`, `issue`*

In [ ]:
%%bigquery
-- your answer here

## Q7. Freshness check  ·  🟢 easy

How fresh is `orders`? Show `latest_order_date` and `days_behind` = days from it to **2025-12-31** (pretend that is today). Show `status` = `STALE` if more than 2 days behind, else `FRESH`.

*Expected result: 1 row · columns: `latest_order_date`, `days_behind`, `status`*

In [ ]:
%%bigquery
-- your answer here

## Q8. Outliers  ·  🟢 easy

Show `order_id`, `customer_id`, `amount` of different orders whose amount is more than **3 times the average** amount.

*Expected result: 6 rows · columns: `order_id`, `customer_id`, `amount`*

In [ ]:
%%bigquery
-- your answer here

## Q9. Values that fail a type cast  ·  🟡 medium

Raw files bring text. From the 4 values `'12'`, `'x7'`, `'3.5'`, `NULL`, show `raw_value` and `as_number` = the number, or NULL when the text is **not a number**. The query must not fail.

*Expected result: 4 rows · columns: `raw_value`, `as_number`*

In [ ]:
%%bigquery
-- your answer here